## CHARTING A COURSE FOR MWANGAZA FILM STUDIO

### PROJECT OBJECTIVES
Help Mwangaza studio determine what kinds of movies are most successful at the box office, using available data.

### RESEARCH OBJECTIVES
Determine Market Viability – Assess which types of films resonate most with audiences and what factors contribute to commercial success.

Guide Strategic Decision-Making – Provide recommendations on which genres, styles, or storytelling approaches the new company should pursue.

To determine the optimal range of movie runtime that is most often associated with high box office success and use it to give insights on the length of an ideal movie.

### LIBRARIES

In [24]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sqlite3
import numpy as np

### DATA CLEANING

In [9]:
#Loading the first dataset
df = pd.read_csv("bom.movie_gross.csv.gz")
df

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010
...,...,...,...,...,...
3382,The Quake,Magn.,6200.0,NaN,2018
3383,Edward II (2018 re-release),FM,4800.0,NaN,2018
3384,El Pacto,Sony,2500.0,NaN,2018
3385,The Swan,Synergetic,2400.0,NaN,2018


In [10]:
#looking for missing values
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values

foreign_gross     1350
domestic_gross      28
studio               5
year                 0
title                0
dtype: int64

In [11]:
df.dropna(subset=["domestic_gross","foreign_gross"],inplace=True)

In [12]:
# Looking for missing values after cleaning
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values

studio            2
year              0
foreign_gross     0
domestic_gross    0
title             0
dtype: int64

In [13]:
df

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010
...,...,...,...,...,...
3275,I Still See You,LGF,1400.0,1500000,2018
3286,The Catcher Was a Spy,IFC,725000.0,229000,2018
3309,Time Freak,Grindstone,10000.0,256000,2018
3342,Reign of Judges: Title of Liberty - Concept Short,Darin Southa,93200.0,5200,2018


In [14]:
# changing the foreign gross column from strings to int and creating a new column with total gross
df["foreign_gross"] = pd.to_numeric(df["foreign_gross"], errors="coerce")
df['total_gross'] = df['domestic_gross'] + df['foreign_gross']
df

,title,studio,domestic_gross,foreign_gross,year,total_gross
0,Toy Story 3,BV,415000000.0,652000000.0,2010,1.067000e+09
1,Alice in Wonderland (2010),BV,334200000.0,691300000.0,2010,1.025500e+09
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000.0,2010,9.603000e+08
3,Inception,WB,292600000.0,535700000.0,2010,8.283000e+08
4,Shrek Forever After,P/DW,238700000.0,513900000.0,2010,7.526000e+08
...,...,...,...,...,...,...
3275,I Still See You,LGF,1400.0,1500000.0,2018,1.501400e+06
3286,The Catcher Was a Spy,IFC,725000.0,229000.0,2018,9.540000e+05
3309,Time Freak,Grindstone,10000.0,256000.0,2018,2.660000e+05
3342,Reign of Judges: Title of Liberty - Concept Short,Darin Southa,93200.0,5200.0,2018,9.840000e+04


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2009 entries, 0 to 3353
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           2009 non-null   object 
 1   studio          2007 non-null   object 
 2   domestic_gross  2009 non-null   float64
 3   foreign_gross   2004 non-null   float64
 4   year            2009 non-null   int64  
 5   total_gross     2004 non-null   float64
dtypes: float64(3), int64(1), object(2)
memory usage: 109.9+ KB


In [16]:
df.describe()

,domestic_gross,foreign_gross,year,total_gross
count,2.009000e+03,2.004000e+03,2009.000000,2.004000e+03
mean,4.697311e+07,7.590713e+07,2013.503235,1.215769e+08
std,8.159966e+07,1.382501e+08,2.598481,2.061554e+08
min,4.000000e+02,6.000000e+02,2010.000000,4.900000e+03
25%,6.650000e+05,3.900000e+06,2011.000000,8.117750e+06
50%,1.650000e+07,1.955000e+07,2013.000000,4.210000e+07
75%,5.600000e+07,7.615000e+07,2016.000000,1.327250e+08
max,9.367000e+08,9.605000e+08,2018.000000,1.518900e+09


In [23]:
#Loading the second dataset
conn = sqlite3.connect("im.db")
cur = conn.cursor()
cur.execute("""SELECT name FROM sqlite_master WHERE type = 'table'""")
table_names = cur.fetchall()
table_names

[]